# Week 4, Day 1: Spatial Data Formats

## Today

By the end of this session you will be able to:

- Compare shapefile, file geodatabase, GeoPackage, and GeoParquet formats by structure, size limits, and use case
- Inspect a dataset's format with `arcpy.Describe` and `arcpy.da.Describe`
- Convert a feature class between formats with `arcpy.conversion.ExportFeatures` and `arcpy.conversion.ExportToParquet`
- Explain why a shapefile's 10-character field-name limit mangles long field names, and to recognize the pattern

## Opener: paired code reading (~15 minutes)

The below Python script constructs a four-county region out of this week's `counties.shp` using five geoprocessing tools, only one of which you've seen so far (in this class). 

Reading code that calls tools you don't know yet is representative of the real world. ***With a buddy:*** read the code below, answering the questions in each block.

### FOLLOW THE RULES!!!


**Rules.**

1. Don't run the cell until block [F] is filled in. Make predictions first
2. Type your answers after the `#` on the blank lines. Keep every line you write starting with `#`
3. **Look things up.** Every block asks about something you probably haven't seen yet. Open the ArcGIS Pro tool reference and read the parameter table. Look at code examples if necessary. 


In [ ]:
import arcpy

arcpy.env.workspace = "YOURPATHGOESHERE/formats_zoo"
arcpy.env.overwriteOutput = True

counties = "counties.shp"
wanted = ["Portage", "Summit", "Cuyahoga", "Kent", "Lake"]

# ── [A] ──────────────────────────────────────────────────────────
# Q: Look up 3734. What coordinate system is it, and what are its units?
#    (HINT: Search "EPSG 3734" or "WKID 3734".) Why would a script that
#    measures anything want this instead of the coordinate system
#    counties.shp is already in?
# A:
#
#
#
ohio_north = arcpy.SpatialReference(3734)


# ── [B] ──────────────────────────────────────────────────────────
# Q: Look up the Select tool (Analysis toolbox) in the Pro tool reference.
#    What is its third parameter called? Why are there single quotes
#    INSIDE the double quotes? What does "NAME = 'Kent'" select from
#    counties.shp, and what does Select write to disk in that case?
# A:
#
#
#
# ── [C] ──────────────────────────────────────────────────────────
# Q: Look up Get Count. What does arcpy.management.GetCount(...) actually
#    return... a number? Why is .getOutput(0) needed, and why the int()
#    wrapped around it? Trace the if/else for "Portage" and for "Kent".
#    Write out what `kept` holds when the loop ends.
# A:
#
#
#
kept = []
for name in wanted:
    out_name = f"pick_{name.lower()}.shp"
    arcpy.analysis.Select(counties, out_name, f"NAME = '{name}'")
    matches = int(arcpy.management.GetCount(out_name).getOutput(0))
    if matches == 1:
        kept.append(out_name)
        print(f"{name}: selected")
    else:
        arcpy.management.Delete(out_name)
        print(f"{name}: {matches} matches, dropped")



# ── [D] ──────────────────────────────────────────────────────────
# Q: Look up Merge and Dissolve (Data Management toolbox). What kind of
#    item does Merge accept as its first parameter, and why does it
#    matter here? In Dissolve, what does "" as the third argument do, and
#    what does [["ALAND", "SUM"]] produce -- what will the new field be
#    called? How many features does region.shp end up with?
# A:
#
#
#
arcpy.management.Merge(kept, "region_parts.shp")
arcpy.management.Dissolve("region_parts.shp", "region.shp", "", [["ALAND", "SUM"]])
arcpy.management.Project("region.shp", "region_ftus.shp", ohio_north)


# ── [E] ──────────────────────────────────────────────────────────
# Q: What list does `kept + ["region_parts.shp", "region.shp"]` build?
#    On a clean run, is the arcpy.Exists() "guard" ever False? So why is
#    it there -- what situation is it protecting against?
# A:
#
#
#
for temp in kept + ["region_parts.shp", "region.shp"]:
    if arcpy.Exists(temp):
        arcpy.management.Delete(temp)

final_count = int(arcpy.management.GetCount("region_ftus.shp").getOutput(0))
print(f"region_ftus.shp: {final_count} feature(s), {ohio_north.name}")


# ── [F] ──────────────────────────────────────────────────────────
# Predict the COMPLETE printed output, top to bottom, one line per line,
# AND list every new file this cell leaves in the folder when it
# finishes. Only run it after you've answered ALL the questions above.
#
#
#
#
#
#
#

## Why formats matter: a contrived story

It's Friday afternoon, you've finished a rush map for a colleague, and you zip up `parks_1mi.shp` and email it to them. 

Monday morning, you receive a reply: "This won't open! ArcGIS says it can't find the attribute table." 

What happened? You sent the `.shp`, but a shapefile isn't one file. It's a bundle. No attribute table --> no working shapefile.

This seems made up (and it is, I suppose). But it also happens ALL THE TIME. 

With respect to spatial data formats: **the format is the container, and the container has rules**.

- how many files it needs
- how long a field name can be
- what data types it can hold
- how big it can get before it breaks

So today we'll use four containers that all hold the exact same 88 Ohio counties

## The file "zoo":

Every file in this week's data folder holds the same 88 Ohio counties. All that is different is the container.

| Property | Shapefile | File geodatabase | GeoPackage | GeoParquet |
|---|---|---|---|---|
| Structure | multiple files per layer (`.shp`, `.shx`, `.dbf`, `.prj`, ...) | one `.gdb` folder holding many datasets | one `.gpkg` file (a SQLite database) | one `.parquet` file (columnar, not row-based) |
| Field name limit | 10 characters | 64 characters | effectively unlimited | effectively unlimited |
| NULL support | no true NULL — the DBF table uses blanks/zeros instead | yes | yes | yes |
| Size cap | 2 GB per file | none in practice (multi-terabyte FGDBs exist) | none in practice | none in practice |
| Standard | de facto industry standard, 1990s-era Esri format | Esri proprietary | OGC open standard | open standard — the GeoParquet specification (developed via OGC) |
| Best at | maximum compatibility with old tools and workflows | fast geoprocessing, domains, subtypes, versioning | single-file sharing outside the Esri world | large tabular analysis, cloud storage, data-science tools |

A few of those rows are worth a sentence each:

- **Shapefile**: still is mostly everywhere, because most software can read one. But it's getting old: multi-file, 2 GB cap, no true NULL (a blank text field and a missing value look identical), and a 10-character field-name limit
- **File geodatabase (FGDB)**: Esri's own container, and the one Pro treats as default for projects. It can hold multiple feature classes plus tables in one folder, supports **domains** (a fixed list of valid values for a field, like an enforced dropdown) and **subtypes** (splitting one feature class into categories with different rules), and geoprocessing tools generally run fastest against it
- **GeoPackage**: a single `.gpkg` file that's actually a SQLite database. An OGC open standard, which means QGIS, R, and many other tools read and write it natively — no Esri license required
- **GeoParquet**: the newest of the four, and most different from the others. **Columnar** (stores each field as its own column-oriented block) rather than row-based, which is what makes it fast for the kind of analysis pandas and geopandas do. Computationally "cheap" to read partially from cloud storage without downloading the whole file

## Your turn ###

With a buddy, use the comparison table above to answer:

***a state agency asks you for county boundaries they can load into QGIS on a laptop with no ArcGIS license, and the file needs to stay small enough to email. Which container should you use? Which should you rule out, and why?***

## Let's write some code:

- First, download the `formats_zoo` data directory, which can be found in `/data/part1`
- Create a new ArcGIS Pro project, then notebook
- Then set your workspace to the `formats_zoo` directory


In [ ]:
import arcpy

zoo_workspace = "C:/Users/pbitterm/Desktop/week0401/formats_zoo"  # your workspace location goes here - change it as appropriate
arcpy.env.workspace = zoo_workspace
arcpy.env.overwriteOutput = True


***What's the expected result?***

## Inspecting a format with `Describe`

Last class, we used `arcpy.ListFields()`. Who can remind me what that does?

The `Describe` function works a bit different - it tells you about the dataset itself, including what format it's stored in, its shape type, its coordinate system, and more. 

There are two versions: 

1. `arcpy.Describe()`: the original, returns an object where you read properties with a dot (`desc.shapeType`)
2. `arcpy.da.Describe()`: the newer one, returns a plain Python **dictionary**, so you read the same properties with brackets (`desc["shapeType"]`) 

This course uses `arcpy.da.Describe()` by default

**Predict before you run:** `counties.shp` is a polygon shapefile. 

Guess what `arcpy.da.Describe("counties.shp")["dataType"]` prints. Is it `"ShapeFile"`, `"FeatureClass"`, or something else?

In [ ]:
shp_desc = arcpy.da.Describe("counties.shp") # what type of object is returned? How would you know? Where would you look?

print(f"dataType: {shp_desc['dataType']}")
print(f"shapeType: {shp_desc['shapeType']}")
print(f"catalogPath: {shp_desc['catalogPath']}")
print(f"spatial reference: {shp_desc['spatialReference'].name}")

**Expected result:**

```
dataType: ShapeFile
shapeType: Polygon
catalogPath: C:\Users\Desktop\pbitterm\week0401\formats_zoo\counties.shp
spatial reference: GCS_North_American_1983
```

A few things:

First, `dataType` for a standalone `.shp` really is `"ShapeFile"`. This is a different value from `"FeatureClass"`, which is what you'll see for the same data if it's inside a geodatabase or GeoPackage. `Describe` distinguishes the container from the geometry type. 

Second, `catalogPath` comes back with backslashes even though you set the workspace with forward slashes. Arcpy normalizes to whatever the operating system expects when it hands a path back to you

**Your turn:** `arcpy.Describe()` (no `da.`) is the older API mentioned above. It returns an object, not a dictionary, so you read the same properties with a dot instead of brackets: `arcpy.Describe("counties.shp").dataType` instead of `arcpy.da.Describe("counties.shp")["dataType"]`. Rewrite the four `print()` lines from the cell above using the dot-access version, and confirm you get the exact same four values

In [ ]:
## Try it here



### A GeoPackage's internal naming

A GeoPackage file can hold more than one layer, so arcpy treats the whole `.gpkg` file itself as a **workspace**. And the individual layers inside it show up the way a SQLite database names its tables.

**Predict before you run:** You're about to set the workspace to `counties.gpkg` itself (not the folder) and list its feature classes. `counties.gpkg` holds one layer, named `counties`. Guess what name shows up in the list. Is it going to be just `"counties"`, or something with an extra prefix?

In [ ]:
arcpy.env.workspace = "counties.gpkg"
gpkg_layers = arcpy.ListFeatureClasses()
print(gpkg_layers)

**Expected result:** `['main.counties']`, not plain `"counties"`. 

A GeoPackage is a SQLite database (that's what "single file, SQLite" in the table above means concretely), and `main` is SQLite's default schema name. Every table gets that prefix automatically. This is the kind of detail `Describe` exists to catch before it surprises you when you're writing your own Python script.

Now describe that layer the same way as before:

In [ ]:
gpkg_desc = arcpy.da.Describe("main.counties")

print(f"dataType: {gpkg_desc['dataType']}")
print(f"shapeType: {gpkg_desc['shapeType']}")

**Expected result:** `dataType: FeatureClass`, `shapeType: Polygon`. `shapeType` matches the shapefile exactly (it's the same polygon geometry, same 88 counties). `dataType` differs: `"FeatureClass"` instead of `"ShapeFile"`, because arcpy now sees the layer sitting inside a container (the GeoPackage) rather than standing alone as a bare file. Again, it's same geometry, same attributes, but different container

**Your turn:** `counties.gpkg` only holds one layer in our example, but a GeoPackage can hold many, each showing up with that same `main.` prefix. While the workspace is still pointed at `counties.gpkg`, run `arcpy.ListFeatureClasses()` again and confirm you get the same one-item list as a few cells ago. 

Then predict: if this GeoPackage held a second layer named `roads`, what would that list return instead?

Switch the workspace back to the folder before the rest of today's cells. If you don't, every plain filename below would resolve inside `counties.gpkg` instead of the `formats_zoo` directory

In [ ]:
arcpy.env.workspace = zoo_workspace
arcpy.env.overwriteOutput = True

**Expected result:** nothing prints. Workspace points where you expect.

### What about the file geodatabase?

There's no `.gdb` sitting in this week's data folder, but the pattern is identical if you point `Describe` at a feature class inside one: `dataType` comes back `"FeatureClass"`, and the geodatabase *itself* describes as `dataType: "Workspace"`. 

But there are some differences: 
- an FGDB feature class can carry a `subtypeFieldName`
- `arcpy.da.ListDomains()` run against the geodatabase can return **domains** (the enforced dropdown-style value lists mentioned in the table above) 

Neither shapefiles nor GeoPackages support either feature

## Converting between formats

`arcpy.conversion.ExportFeatures` is the modern, general-purpose "save a copy of this dataset, optionally in a different format" tool. Point it at an input file, provide an output path in the container you want, and it writes a full copy: same fields, same geometry, same row count

**Predict before you run:** You're about to export `counties.shp` into a brand-new GeoPackage, `counties_converted.gpkg`, as a layer named `counties`. Guess how many rows the new layer has, and whether any field names change along the way.

In [ ]:
arcpy.management.CreateSQLiteDatabase("counties_converted", spatial_type="GEOPACKAGE")
# creates counties_converted.gpkg in the current workspace

arcpy.conversion.ExportFeatures("counties.shp", "counties_converted.gpkg/counties") # note the syntax for the output path - it is a combination of the GPKG file name and the layer name
print("export complete")

**Expected result:** `export complete` prints, and a new file, `counties_converted.gpkg`, appears in the folder with one layer, `counties`, inside it. 

Row count: 88, matching the source exactly.  Notice the output path itself follows the same pattern as the GeoPackage's internal naming from a few cells ago: `<file>.gpkg/<layer name>`.

**Predict before you run:** This dataset's own field names (`STATEFP`, `COUNTYNS`, `NAMELSAD`, and so on) all happen to be 10 characters or shorter already. Guess whether `arcpy.ListFields()` on the new GeoPackage layer shows any names different from the original shapefile's.

In [ ]:
arcpy.env.workspace = "counties_converted.gpkg"
converted_fields = [f.name for f in arcpy.ListFields("main.counties")]
print(converted_fields)

arcpy.env.workspace = zoo_workspace

**Expected result:** the same field names as `counties.shp`: `STATEFP`, `COUNTYFP`, `COUNTYNS`, `GEOID`, `GEOIDFQ`, `NAME`, `NAMELSAD`, `LSAD`, `CLASSFP`, `MTFCC`, `CSAFP`, `CBSAFP`, `METDIVFP`, `FUNCSTAT`, `ALAND`, `AWATER`, `INTPTLAT`, `INTPTLON`, plus arcpy's own `OBJECTID` and `Shape` fields. 

Nothing changed, because nothing *had* to... every one of this dataset's field names already fits inside a shapefile's 10-character limit. But that's a property of this particular Census-derived dataset, not a guarantee

**Your turn:** Export `counties.shp` again, this time into a brand-new shapefile copy instead of a GeoPackage: `arcpy.conversion.ExportFeatures("counties.shp", "counties_copy.shp")`. Same tool, same-format output this time. 

Predict first: given every field name here already fits inside 10 characters either way, does the field-name behavior differ at all from the GeoPackage export above?


In [ ]:
## Try it here

### A version caveat: exporting to GeoParquet

ArcGIS Pro has a dedicated conversion tool for this: **Export To Parquet** (`arcpy.conversion.ExportToParquet(inputs, out_file)`), which writes a point, line, or polygon feature class (or a plain table) to an Apache Parquet file. Try that first — `arcpy.conversion.ExportToParquet("counties.shp", "counties_from_shp.parquet")`. **It's a newer addition to the Conversion toolbox, so you need at least ArcGIS Pro 3.6**



In [ ]:
## Try it here (if you're at 3.6 or above)



## Field-name truncation: the ten-character wall

Every field name in this week's counties data already fits inside a shapefile's limit, which is  why the zoo files above didn't show you a truncation problem. But Real-world data is not always so polite. American Community Survey (ACS) extracts, for instance, ship with field names built to be descriptive to a human, not short enough for a 1990s database format:

| Original field name | Length | Shapefile field name (first 10 characters) |
|---|---|---|
| `median_household_income` | 23 | `median_hou` |
| `pct_bachelor_degree_or_higher` | 29 | `pct_bachel` |
| `total_population_age_65_plus` | 28 | `total_popu` |
| `housing_units_built_before_1970` | 31 | `housing_un` |

Worse, two long names that happen to share their first ten characters — `median_household_income` and `median_household_size`, say — truncate to the exact same name, `median_hou`, which a shapefile doesn't allow twice. Whatever tool wrote the file resolves the collision by silently appending a digit, so you'd end up with `median_hou` and `median_ho1` side by side in an attribute table, with nothing in the field name itself telling you which is income and which is household size.



**Your turn:** `precipitation_station_id` is a plausible real-world field name for weather data. Using Python's slicing (`"precipitation_station_id"[:10]`), predict its shapefile-truncated form by hand, then check your answer in a code cell. 

Then invent a second field name, at least 15 characters long, that would collide with it after truncation — what would arcpy's silent-digit-append pattern from the table above turn that pair into?

In [ ]:
## Try it here



## Choosing a format

No single format is always correct. A few decision rules that cover most of what you'll run into in this course and beyond:

- **Handing data to an old tool, or a colleague on a workflow you don't control?** Shapefile (annoying, but common) 
- **Building something inside ArcGIS Pro that needs domains, subtypes, versioning, or just needs to run fast on a big multi-layer project?** File geodatabase
- **Sharing one file with someone who isn't necessarily using Esri software (e.g., QGIS, R, a web map library)?** GeoPackage — an open standard, one file, no missing sidecars
- **Loading data into pandas or geopandas for analysis, or archiving something read-mostly in cloud storage?** GeoParquet

Notice: no cases are "always use X." Choose a format based on what has to happen to the data next.

### Format decision mini-case: the 3 GB shapefile

A colleague at a county GIS office emails you a folder named `parcels.shp`. Except when you go looking, "the shapefile" turns out to be six files (`.shp`, `.shx`, `.dbf`, `.prj`, `.cpg`, `.sbn`) totaling just under 3 GB, covering every parcel in the county going back a decade. Two things should stop you immediately, using nothing but what you learned today:

- **The 2 GB cap.** A single shapefile can't actually hold that. What almost certainly happened: the county's GIS software silently split it — you're probably looking at `parcels.shp`, `parcels2.shp`, `parcels3.shp`, and so on, each safely under the cap, described to you casually as "the shapefile" (singular) by someone who deals with this split every day and stopped noticing it. Before you write a single line of code, count how many `.shp` files are actually sitting in that folder
- **The sidecar risk, multiplied.** One shapefile is already a five-or-six-file bundle where losing just the `.dbf` breaks everything. Multiply that by however many split pieces you actually found, and the odds that *something* goes missing in an email attachment or a shared-drive copy climb fast.


***With a buddy:***  What would you convert it to, and why? 

### THen:
Using the same four decision-rule bullets above, work through your own one-paragraph mini-case: your PI wants next semester's field data version-controlled in a shared repo, edited by three different people during the collection season, then merged into one dataset for analysis once the semester ends. Which format for the editing season, which format (if different) for the merged analysis copy, and why?



## Sketch 2: Format Detective

Today's format zoo is the focus of Sketch 2. Starting from the shapefile specified in the sketch, you'll convert it to GeoPackage (`arcpy.conversion.ExportFeatures`) and to GeoParquet (`arcpy.conversion.ExportToParquet`) - you may need to use the other lab or a personal install. The,  build a short comparison table: file size, longest field name allowed, and two comparison points of your own choosing.

**Try it yourself:**

1. Try `arcpy.da.Describe("counties.parquet")` on the file already in this folder. Compare `dataType` to what you saw for `counties.shp` and `main.counties` today.
2. Pick one of the four ACS-style field names from the truncation table and confirm by hand that Python's `[:10]` slicing matches the "shapefile field name" column.
3. Check the file size of `counties.shp`, `counties.gpkg`, and `counties.parquet` on disk (Windows Explorer's file properties, or `os.path.getsize()` if you're scripting it).




## For next class:

- Lab 1 due this week
- Sketch 2 due Thursday, peer review follows
- Readings on Canvas
- Lab 2 starts Thursday